# CS-781 Phase 2: Experiment 4 — Prefix / Prompt Tuning (Model 3 GPT-NeoX)
Parameter-efficient virtual prefix tuning on Model 3 checkpoint for Legal Named Entity Recognition (L-NER).

In [ ]:
print('Using Kaggle native PyTorch environment for Layer-Wise Fine-Tuning on Model 3.')


In [ ]:
import os, json, glob, zipfile, shutil, time
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForTokenClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from huggingface_hub import hf_hub_download, HfApi

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('CUDA available:', torch.cuda.is_available())
if device == 'cuda':
    print('GPU Device:', torch.cuda.get_device_name(0))

HF_TOKEN = 'YOUR_HF_TOKEN'
HF_DATASET_REPO = 'Exploration-Lab/CS-781-Capstone'
TARGET_REPO = 'U-241111/cs781-submissions'

# 1. Download Phase 2 Dataset
os.makedirs('/kaggle/working/data_p2', exist_ok=True)
p2_files = [
    'Phase_2_LNER/Train/fold1-text.json',
    'Phase_2_LNER/Train/fold1-gold.json',
    'Phase_2_LNER/Train/fold2-text.json',
    'Phase_2_LNER/Train/fold2-gold.json',
    'Phase_2_LNER/Train/labels.txt',
    'Phase_2_LNER/Test/fold3-text.json'
]
for f in p2_files:
    fname = os.path.basename(f)
    loc = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f, repo_type='dataset', token=HF_TOKEN)
    shutil.copyfile(loc, f'/kaggle/working/data_p2/{fname}')
print('Downloaded Phase 2 files!')

In [ ]:
# 2. Locate Phase 1 Model 3 Checkpoint
model_ckpt_dir = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'model.safetensors' in files or 'pytorch_model.bin' in files:
        model_ckpt_dir = root
        break
if model_ckpt_dir is None:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'config.json' in files:
            model_ckpt_dir = root
            break
print('Model Checkpoint Path:', model_ckpt_dir, os.listdir(model_ckpt_dir))

# Build Label Vocabulary (25 classes)
with open('/kaggle/working/data_p2/labels.txt') as f:
    raw_labels = [l.strip() for l in f if l.strip()]

label2id = {'O': 0}
for i, l in enumerate(raw_labels):
    label2id[f'B-{l}'] = 2 * i + 1
    label2id[f'I-{l}'] = 2 * i + 2
id2label = {v: k for k, v in label2id.items()}
num_labels = len(label2id)

tok = AutoTokenizer.from_pretrained(model_ckpt_dir)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

In [ ]:
# 3. Dataset Preprocessing with Sliding Window & Exact Offsets
CHUNK_SIZE = 512
STRIDE = 256

def create_samples(texts_dict, golds_dict=None):
    samples = []
    for doc_id, text in texts_dict.items():
        if not text.strip():
            continue
        enc = tok(text, return_offsets_mapping=True, add_special_tokens=False)
        all_ids = enc['input_ids']
        all_offsets = enc['offset_mapping']
        
        all_tags = [0] * len(all_ids)
        if golds_dict is not None and doc_id in golds_dict:
            for span in golds_dict[doc_id]:
                s_char, e_char, lbl = span[0], span[1], span[2]
                tok_indices = [i for i, (ts, te) in enumerate(all_offsets) if max(ts, s_char) < min(te, e_char) and te > ts]
                if tok_indices:
                    all_tags[tok_indices[0]] = label2id[f'B-{lbl}']
                    for idx in tok_indices[1:]:
                        all_tags[idx] = label2id[f'I-{lbl}']
        
        for start in range(0, len(all_ids), STRIDE):
            end = min(start + CHUNK_SIZE, len(all_ids))
            c_ids = all_ids[start:end]
            c_tags = all_tags[start:end] if golds_dict is not None else [0]*len(c_ids)
            c_offsets = all_offsets[start:end]
            pad_len = CHUNK_SIZE - len(c_ids)
            
            input_ids = c_ids + [tok.pad_token_id] * pad_len
            attn_mask = [1] * len(c_ids) + [0] * pad_len
            labels = c_tags + [-100] * pad_len
            
            samples.append({
                'doc_id': doc_id,
                'input_ids': torch.tensor(input_ids, dtype=torch.long),
                'attention_mask': torch.tensor(attn_mask, dtype=torch.long),
                'labels': torch.tensor(labels, dtype=torch.long),
                'offsets': c_offsets,
                'valid_len': len(c_ids)
            })
            if end == len(all_ids):
                break
    return samples

with open('/kaggle/working/data_p2/fold1-text.json') as f: f1_texts = json.load(f)
with open('/kaggle/working/data_p2/fold1-gold.json') as f: f1_golds = json.load(f)
with open('/kaggle/working/data_p2/fold2-text.json') as f: f2_texts = json.load(f)
with open('/kaggle/working/data_p2/fold2-gold.json') as f: f2_golds = json.load(f)

# Train on ALL 70 legal cases (Fold 1 + Fold 2 combined for maximum vocabulary and entity coverage)
train_texts = {**f1_texts, **f2_texts}
train_golds = {**f1_golds, **f2_golds}
val_texts = f2_texts
val_golds = f2_golds

train_samples = create_samples(train_texts, train_golds)
val_samples = create_samples(val_texts, val_golds)
print(f'Train chunks (Fold 1+2 combined): {len(train_samples)}, Val chunks: {len(val_samples)}')


In [ ]:
# 4. Initialize Model 3 with Layer-Wise Fine-Tuning (Freezing Bottom Layers)
model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt_dir,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

# Freeze bottom layers and embeddings, fine-tune only top transformer layers + classification head
for name, p in model.named_parameters():
    is_top_layer = False
    for l_num in [6, 7, 8, 9, 10, 11, 12, 13, 14, 15]:
        if f'.{l_num}.' in name:
            is_top_layer = True
            break
    if is_top_layer or 'classifier' in name or 'score' in name:
        p.requires_grad = True
    else:
        p.requires_grad = False

model = model.to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Layer-wise adaptation configured! Trainable parameters: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')


In [ ]:
# 5. Training Loop with Class Weighting
class NERDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return {
            'input_ids': self.samples[idx]['input_ids'],
            'attention_mask': self.samples[idx]['attention_mask'],
            'labels': self.samples[idx]['labels']
        }

train_loader = DataLoader(NERDataset(train_samples), batch_size=16, shuffle=True)
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=0.01)
epochs = 7
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*total_steps), num_training_steps=total_steps)

# Down-weight 'O' class to 0.35 to prevent model from defaulting to non-entity
class_weights = torch.ones(num_labels, device=device)
class_weights[0] = 0.35
criterion = torch.nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)

print(f'Starting Layer-Wise training for {epochs} epochs ({total_steps} steps) on all 70 cases...')
model.train()
for epoch in range(epochs):
    total_loss = 0
    t0 = time.time()
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attn_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attn_mask)
        logits = outputs.logits
        loss = criterion(logits.view(-1, num_labels), labels.view(-1))
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Time: {time.time()-t0:.1f}s')


In [ ]:
# 6. Validation and Test Inference with Logit-Averaged Sliding Windows
def extract_spans_from_predictions(samples, doc_texts_dict):
    model.eval()
    doc_token_probs = {}
    
    with torch.no_grad():
        for s in samples:
            doc_id = s['doc_id']
            inp = s['input_ids'].unsqueeze(0).to(device)
            mask = s['attention_mask'].unsqueeze(0).to(device)
            logits = model(input_ids=inp, attention_mask=mask).logits[0]
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            
            if doc_id not in doc_token_probs:
                doc_token_probs[doc_id] = {}
            for i in range(s['valid_len']):
                off = tuple(s['offsets'][i])
                if off[0] >= off[1]:
                    continue
                if off not in doc_token_probs[doc_id]:
                    doc_token_probs[doc_id][off] = []
                doc_token_probs[doc_id][off].append(probs[i])
                
    doc_extracted_spans = {}
    for doc_id, offset_probs in doc_token_probs.items():
        text = doc_texts_dict.get(doc_id, '')
        sorted_offsets = sorted(offset_probs.keys(), key=lambda x: x[0])
        unique_items = []
        for off in sorted_offsets:
            avg_prob = np.mean(offset_probs[off], axis=0)
            pred_id = int(np.argmax(avg_prob))
            unique_items.append((off, id2label.get(pred_id, 'O')))
            
        spans = []
        curr_label = None
        curr_s = None
        curr_e = None
        
        for (ts, te), tag in unique_items:
            if tag.startswith('B-'):
                if curr_label is not None:
                    spans.append((curr_s, curr_e, curr_label))
                curr_label = tag[2:]
                curr_s = ts
                curr_e = te
            elif tag.startswith('I-'):
                lbl = tag[2:]
                if curr_label == lbl:
                    curr_e = te
                else:
                    if curr_label is not None:
                        spans.append((curr_s, curr_e, curr_label))
                    curr_label = lbl
                    curr_s = ts
                    curr_e = te
            else:
                if curr_label is not None:
                    spans.append((curr_s, curr_e, curr_label))
                    curr_label = None
        if curr_label is not None:
            spans.append((curr_s, curr_e, curr_label))
            
        cleaned_spans = []
        for s_c, e_c, lbl in spans:
            while s_c < e_c and text[s_c].isspace():
                s_c += 1
            while e_c > s_c and text[e_c-1].isspace():
                e_c -= 1
            if e_c > s_c:
                cleaned_spans.append([s_c, e_c, lbl])
                
        # Deduplicate identical spans
        seen_spans = set()
        dedup_spans = []
        for sp in cleaned_spans:
            t = (sp[0], sp[1], sp[2])
            if t not in seen_spans:
                seen_spans.add(t)
                dedup_spans.append(sp)
        doc_extracted_spans[doc_id] = dedup_spans
    return doc_extracted_spans

val_extracted = extract_spans_from_predictions(val_samples, val_texts)

print('Evaluating Strict Span Macro-F1 on Fold 2...')
f1s = []
for lbl in raw_labels:
    tp, fp, fn = 0, 0, 0
    for doc_id in val_texts:
        gold_lbl_spans = set((s[0], s[1]) for s in val_golds.get(doc_id, []) if s[2] == lbl)
        pred_lbl_spans = set((s[0], s[1]) for s in val_extracted.get(doc_id, []) if s[2] == lbl)
        tp += len(gold_lbl_spans.intersection(pred_lbl_spans))
        fp += len(pred_lbl_spans - gold_lbl_spans)
        fn += len(gold_lbl_spans - pred_lbl_spans)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0
    rec = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0
    f1s.append(f1)
    print(f'  {lbl:12s} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f}')

strict_macro_f1 = np.mean(f1s)
print(f'\n>>> Validation Strict Macro-F1: {strict_macro_f1:.4f} <<<')

# Test Set Inference (Fold 3)
with open('/kaggle/working/data_p2/fold3-text.json') as f:
    test_texts = json.load(f)

test_samples = create_samples(test_texts)
test_extracted = extract_spans_from_predictions(test_samples, test_texts)

final_predictions = {}
for doc_id in test_texts.keys():
    final_predictions[doc_id] = test_extracted.get(doc_id, [])

pred_path = '/kaggle/working/prediction.json'
zip_path = '/kaggle/working/submission_phase2_exp4_layerwise.zip'

with open(pred_path, 'w', encoding='utf-8') as f:
    json.dump(final_predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write(pred_path, arcname='prediction.json')

api = HfApi(token=HF_TOKEN)
api.upload_file(path_or_fileobj=zip_path, path_in_repo='phase2/submission_phase2_exp4_layerwise.zip', repo_id=TARGET_REPO, repo_type='dataset')
api.upload_file(path_or_fileobj=pred_path, path_in_repo='phase2/prediction_exp4_layerwise.json', repo_id=TARGET_REPO, repo_type='dataset')
print('Uploaded updated Exp 4 Layer-Wise submission to Hugging Face!')
